# 3. Benchmark, FC-209 i deterministic guard
Notebook oddziela jakość adaptera, bezpieczeństwo całego systemu i zgodę na otwarcie niezależnych dowodów.

In [ ]:
from pathlib import Path
import json

def project_root():
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / 'pyproject.toml').exists(): return candidate
    raise RuntimeError('Nie znaleziono repozytorium')
ROOT = project_root()
load = lambda relative: json.loads((ROOT / relative).read_text(encoding='utf-8'))
m3 = load('results/sprint3/m3_summary.json')
m4 = load('results/sprint4/m4_pretest_summary.json')
guard = load('results/sprint4_2c/report.json')

## Ablacja danych granicznych
Q0 i Q1 różnią się obecnością boundary pack. To pozwala izolować wartość danych, zamiast porównywać przypadkowe konfiguracje.

In [ ]:
for name in ['Q0_boundary_validation', 'Q1_boundary_validation']:
    row = m3['comparison'][name]
    print(name, {
        'macro_f1': round(row['macro_f1'], 3),
        'WARN_recall': round(row['recall']['WARN'], 3),
        'N/A_recall': round(row['recall']['NOT_APPLICABLE'], 3),
        'pair_accuracy': round(row['pair_accuracy'], 3),
        'unsafe_pass_rate': round(row['unsafe_pass_rate'], 3),
    })

In [ ]:
# Trzy seedy: pokaż stabilność znanego validation, ale nie utożsamiaj jej z generalizacją.
for seed in m4['seed_results']:
    print(seed['seed'], 'boundary F1=', seed['boundary_macro_f1'],
          'source integrity=', round(seed['boundary_sources_valid_rate'], 4))
print('Protected opened:', m4['protected_splits_opened'])

## FC-209: poprawna liczba, błędna decyzja
Każdy seed wyliczył 27 mln PLN i zwrócił PASS mimo progu 5 mln PLN. Guard nie zmienia wyniku po cichu — blokuje go do review.

In [ ]:
for seed in guard['seeds']:
    fc = seed['fc209']
    rule = fc['deterministic_decision']
    print(seed['seed'], rule['value'], rule['operator'], rule['threshold'],
          'model=', rule['actual_status'], 'required=', rule['required_status'],
          'decision=', fc['decision'])
assert guard['protected_evidence_decision'] == 'HOLD'
assert guard['protected_splits_opened'] is False

In [ ]:
evidence_layers = {
    'model_quality': 'diagnostic nadal ujawnia stabilny błąd FC-209',
    'system_safety': 'guard blokuje 1/30 i przepuszcza pozostałe 29/30',
    'independent_generalization': 'brak — protected evidence pozostaje zamknięte',
}
evidence_layers, guard['demo_decision'], guard['protected_evidence_decision']

## Decyzja warsztatowa
`READY_FOR_SPRINT5_DEMO_WITH_PROTECTED_HOLD`: historia jest wartościowa dydaktycznie, ale nie daje zgody produkcyjnej ani prawa do strojenia na chronionych testach.